In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as opt
import matplotlib.pyplot as plt
import numpy as np
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import wandb
import json

with open('secrets.json', 'r') as f:
    secrets = json.load(f)

wandb.login(key=secrets['WANDB_API_KEY'])

wb_key='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'
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")



PATH = './cifar_net.pth'

batch_size=64

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

train_dataset = datasets.MNIST(root='./data', train=True,  download=True, transform=transform)
test_dataset  = datasets.MNIST(root='./data', train=False, download=True, transform=transform)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True,  num_workers=2)
test_loader  = DataLoader(test_dataset,  batch_size=batch_size, shuffle=False, num_workers=2)

images, labels = next(iter(train_loader))
print(images.shape)  # torch.Size([64, 1, 28, 28])
print(labels.shape)  # torch.Size([64])


In [ ]:
class Net(nn.Module):
    def __init__(self,use_batchnorm=False):
        super(Net,self).__init__()
        self.conv1=nn.Conv2d(1,10,3)
        self.bn1=nn.BatchNorm2d(10) if use_batchnorm else nn.Identity()
        self.conv2=nn.Conv2d(10,20,3)
        self.bn2=nn.BatchNorm2d(20) if use_batchnorm else nn.Identity()

        self.fc1=nn.Linear(500,144)
        self.fc2=nn.Linear(144,72)
        self.fc3=nn.Linear(72,10)
    
    def forward(self,input):
        
        c1=F.relu(self.conv1(input))
        c1=self.bn1(c1)        
        s2=F.max_pool2d(c1,(2,2))
        c3=F.relu(self.conv2(s2))
        c3=self.bn2(c3)
        s4=F.max_pool2d(c3,2)
        s4=torch.flatten(s4,1)
        f5=F.relu(self.fc1(s4))
        f6=F.relu(self.fc2(f5))
        output=self.fc3(f6)
        return output



In [ ]:


epoch=50
lr=1e-1

config={
    'epoch':epoch,
    'lr':lr
}

net=Net().to(device)
print(net)       

criterion=nn.CrossEntropyLoss()
optmizer=opt.Adam(net.parameters(),lr=lr)

def val_acc(net,test_loader):
    correct=0
    total=0
    net.eval()
    with torch.no_grad():
        for data in test_loader:
            images,labels=data
            images, labels = images.to(device), labels.to(device)
            outputs=net(images)
            _,predicted=torch.max(outputs,1)
            total+=labels.size(0)
            correct+=(predicted == labels).sum().item()
    net.train()
    return correct/total

wandb.init(project='lenet5exp',name=f'lr={lr}_Bn=F',config=config)
for epo in range(epoch):
    net.train()
    running_loss=0.0
    for i,data in enumerate(train_loader,0):
        inputs,labels=data
        inputs, labels = inputs.to(device), labels.to(device)
        optmizer.zero_grad()
        outputs=net(inputs)
        loss=criterion(outputs,labels)
        loss.backward()
        optmizer.step()

        running_loss+=loss.item()
        if i%20 == 19:
            print(f'[{epo+1},{i+1:5d}] loss={running_loss/20:.3f}')
    acc=val_acc(net,test_loader)
    wandb.log({"accuracy": acc, "loss": running_loss/len(train_loader)})
wandb.finish()
print('finish train')
torch.save(net.state_dict(), PATH)

In [ ]:
net.load_state_dict(torch.load(PATH, weights_only=True))
def val_acc():
    correct=0
    total=0
    with torch.no_grad():
        for data in test_loader:
            images,labels=data
            outputs=net(images)
            _,predicted=torch.max(outputs,1)
            total+=labels.size(0)
            correct+=(predicted == labels).sum().item()
    return correct//total
